# EDA датасета и первая предобработка

### О датасете

Датасет основан на **TinyGSM** и содержит математические вопросы с программами решения.

- **Train:** 11 838 003 примера в 60 файлах Parquet.
- **Test:** 1 919 вопросов без эталонных программ и числовых ответов.
- **Предобработка:** из программ удалены docstring, основная функция переименована в `solve`.

Числовые ответы, идентификаторы и количество токенов не сохранены. Эталонные программы не запускались, поэтому в train могут встречаться решения, которые завершаются ошибкой или не возвращают число.

### Загрузка датасета

Проверка HF_token'а:

In [5]:
import sys
from pathlib import Path

project_dir = Path.cwd()
if not (project_dir / "lib").is_dir():
    project_dir = project_dir.parent
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

from lib.utils.paths import PROJECT_DIR

from huggingface_hub import whoami

print(whoami()["name"])

62wip


Теперь загрузим данные:

In [ ]:
from datasets import load_dataset

ds = load_dataset("SlavaYus/tinysms_qa_12m")
print(ds["train"][0], len(ds["test"]))

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/60 [00:00<?, ?it/s]

data/train/part-00045.parquet: reconstructing file:   0%|          |  0.00B / 23.5MB            

data/train/part-00045.parquet: downloading bytes:           |  0.00B            

data/train/part-00046.parquet: reconstructing file:   0%|          |  0.00B / 23.6MB            

...

data/test/test.parquet: reconstructing file:   0%|          |  0.00B /  136kB            

data/test/test.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/11838003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1919 [00:00<?, ? examples/s]

{'question': 'Mark has 10 crayons. He gives 2 crayons to his younger sister and loses another 4 while he was playing. How many crayons does Mark have left?', 'code': 'def solve() -> int:\n    crayonsTotal = 10\n    crayonsGiven = 2\n    crayonsLost = 4\n    crayonsLeft = crayonsTotal - crayonsGiven - crayonsLost\n    result = crayonsLeft\n\n    return result\n'} 1919


In [6]:
ds

DatasetDict({
    train: Dataset({
        features: ['question', 'code'],
        num_rows: 11838003
    })
    test: Dataset({
        features: ['question', 'code'],
        num_rows: 1919
    })
})

In [14]:
train_pd = ds["train"].to_pandas()

In [26]:
train_pd.sample(2)

,question,code
5774045,"In a school with 80 students, 35% are girls. I...",def solve() -> int:\n # Given data\n tot...
11594400,Tonya sold 100 potatoes at the farmer's market...,def solve() -> int:\n potatoes = 100\n f...


Читается тяжело, попробуем вывести полноценно. Для этого соберем какую-то функцию `pretty_show_example`:

In [32]:
from html import escape
from textwrap import dedent
from IPython.display import HTML, Code, display
from pygments import highlight
from pygments.lexers import PythonLexer
from pygments.formatters import HtmlFormatter



def pretty_show_example(example):
    question = escape(example["question"].strip())
    code = dedent(example["code"]).strip("\n")

    display(HTML(f"""
        <div style="
            box-sizing: border-box;
            max-width: 100%;
            padding: 18px 22px;
            margin: 8px 0;
            border: 1px solid #cbd5e1;
            border-radius: 12px;
            background: #f8fafc;
            color: #0f172a;
        ">
            <div style="
                font-weight: 600;
                color: #64748b;
                margin-bottom: 10px;
            ">Вопрос:</div>

            <div style="
                font-size: 16px;
                line-height: 1.6;
                white-space: normal;
                overflow-wrap: anywhere;
            ">{question}</div>
        </div>

        <h4 style="margin: 18px 0 8px;">Решение:</h4>
    """))

    formatter = HtmlFormatter(
        noclasses=True,
        cssstyles="color: #111827; padding: 14px; border-radius: 10px;",
        prestyles="color: inherit; background: transparent; margin: 0;",
    )
    display(HTML(highlight(code, PythonLexer(), formatter)))

In [33]:
train = ds["train"]
pretty_show_example(train[67])

Другое дело, теперь можно приятно смотреть на примеры. Посмотрим на какую-то пачку примеров глазами:

In [45]:
seed = 62  # Запинем сид чтобы везде был один и тот же рандом

In [46]:
from random import Random

indices = Random(seed).sample(range(len(train)), 7)


for n, idx in enumerate(indices, start=1):
    display(HTML(f"<h3>Пример №{n} - строка {idx:,}</h3>"))
    pretty_show_example(train[idx])

Видно, что в решениях часто присутствуют пояснительные комментарии, что не есть хорошо для обучения наверное. Вернемся к ним попозже, мб вообще будем экспить с ними.

#### Первые статистики по данным

Посмотрим на распределение длин вопросов и кода

In [47]:
train_pd["question_length"] = train_pd["question"].str.len()
train_pd["code_length"] = train_pd["code"].str.len()

train_pd[["question_length", "code_length"]].describe(
    percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
).round(1)

,question_length,code_length
count,11838003.0,11838003.0
mean,191.9,258.9
std,79.3,135.2
min,11.0,20.0
25%,133.0,170.0
50%,176.0,225.0
75%,235.0,305.0
90%,297.0,417.0
95%,341.0,514.0
99%,441.0,783.0


Посмотрим на пару самых маленьких и самых больших строк по кол-ву символов.

In [49]:
for column in ["question_length", "code_length"]:
    lengths = train_pd[column]
    threshold = lengths.quantile(0.99)
    tail = lengths[lengths > threshold]

    groups = {
        "Сразу выше 99-го перцентиля": tail.nsmallest(2),
        "Случайные примеры из хвоста": tail.sample(2, random_state=seed),
        "Самые длинные примеры": lengths.nlargest(2),
    }

    for title, examples in groups.items():
        display(HTML(f"<h3>{column}: {title}</h3>"))

        for idx, length in examples.items():
            display(HTML(
                f"<b>Строка {idx:,} · {length:,} символов</b>"
            ))
            pretty_show_example(train_pd.loc[idx])

**Выводы по распределениям длин:**

**Длина вопросов:** По просмотренным примерам 99-й перцентиль как жёсткую границу фильтрации не стоит: выше него встречаются содержательные вопросы с корректными на первый взгляд решениями. Это и примеры прям над границей и рандомные из хвоста.

При этом среди самых длинных вопросов обнаружены прям плохие примеры. В одном объединены несколько несвязанных условий, а решение не соответствует вопросу. Эту строку стоит исключить. Это даёт основание рассмотреть ограничение на максимальную длину вопроса.

**Длина кода:** Отсечение по 99-му перцентилю также приведёт к потере корректных примеров. Однако некоторые крайние значения объясняются многословными комментариями, огромными последовательностями пробелов или длинным принтом. Поэтому большая длина кода может отражать особенности форматирования, а не сложность решения и скорее содержит мусор.

В качестве предварительных трешхолдов для фильтрации хочу взять какие-то абсолютные числа.

In [51]:
for column, threshold in [
    ("question_length", 1000),
    ("code_length", 1200)
]:
    lengths = train_pd[column]
    tail = lengths[lengths > threshold]


    display(HTML(f"<h3>{column}: 2 примера прям над трешхолдом</h3>"))

    for idx, length in tail.nsmallest(2).items():
        display(HTML(
            f"<b>Строка {idx:,} · {length:,} символов</b>"
        ))
        pretty_show_example(train_pd.loc[idx])

Остановимся на этих числах. Почистим данные:

In [52]:
drop_mask = (
    (train_pd["question_length"] > 1000)
    | (train_pd["code_length"] > 1200)
)

n_total = len(train_pd)
n_drop = int(drop_mask.sum())

print(f"Всего примеров: {n_total:,}")
print(f"Отсекаем: {n_drop:,} ({n_drop / n_total:.2%})")
print(f"Остаётся: {n_total - n_drop:,}")

Всего примеров: 11,838,003
Отсекаем: 7,009 (0.06%)
Остаётся: 11,830,994


In [53]:
rejected_sample = train_pd.loc[drop_mask].sample(
    n=min(10, n_drop),
    random_state=seed,
)

for idx, example in rejected_sample.iterrows():
    display(HTML(
        f"<h3>Строка {idx:,} · "
        f"вопрос: {example['question_length']:,} символов · "
        f"код: {example['code_length']:,} символов</h3>"
    ))
    pretty_show_example(example)

Тут больше косяков, чем норм выводов, поэтому верим, что отсечки норм.

In [54]:
train_clean = train_pd.loc[~drop_mask].copy()

### Комментарии в коде

При просмотре примеров легко заметить, что во многих решениях присутствуют пояснительные комментарии. Хочется оценить, какая доля кода их содержит и насколько их удаление сократит длину решений в символах. 

Для этого возьмем семпл из 1 000 000 примеров из выборки и сравним длины до и после удаления комментариев. Если останется время и эксперимент покажется целесообразным, сравним два варианта обучения: с комментариями в решениях и без них.

In [62]:
import tokenize
from io import StringIO


def remove_comments(code):
    lines = code.split("\n")

    for token in tokenize.generate_tokens(StringIO(code).readline):
        if token.type == tokenize.COMMENT:
            row, column = token.start
            line = lines[row - 1]

            before = line[:column].rstrip()
            after = line[token.end[1]:]

            lines[row - 1] = before + after if before else None

    return "\n".join(line for line in lines if line is not None)

In [64]:
import pandas as pd
from tqdm.auto import tqdm

comment_sample = train_clean.sample(n=1_000_000, random_state=seed).copy()

tqdm.pandas(desc="Удаляем комментарии")
comment_sample["code_no_comments"] = (
    comment_sample["code"].progress_apply(remove_comments)
)

comment_sample["has_comments"] = (
    comment_sample["code"] != comment_sample["code_no_comments"]
)

print(f"С комментариями: {comment_sample['has_comments'].mean():.2%}")

lengths = pd.DataFrame({
    "before": comment_sample["code"].str.len(),
    "after": comment_sample["code_no_comments"].str.len(),
})
lengths["removed"] = lengths["before"] - lengths["after"]

reduction = lengths["removed"].sum() / lengths["before"].sum()
print(f"Сокращение объёма символов: {reduction:.2%}")

lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1)

Удаляем комментарии:   0%|          | 0/1000000 [00:00<?, ?it/s]

С комментариями: 31.69%
Сокращение объёма символов: 9.28%


,before,after,removed
count,1000000.0,1000000.0,1000000.0
mean,258.5,234.5,24.0
std,132.8,98.9,72.7
min,55.0,53.0,0.0
50%,225.0,213.0,0.0
90%,416.0,364.0,67.0
95%,513.0,425.0,174.0
99%,775.0,564.0,370.0
max,1200.0,1187.0,1059.0


Я предлагаю дропнуть комменты из строк как бейзлайн при обучении. Для генерации такое сокращение потенциально полезно: модель будет выдавать меньше пояснительного текста и больше учиться писать код. 

Но на диск сохраним оба вида текстов (вероятно не в этом ноутбуке): базовые и чищенные. Это позволит потом легко переключаться при обучении.

In [65]:
from tqdm.auto import tqdm

tqdm.pandas(desc="Удаляем комментарии во всём train_clean")
train_clean["code_no_comments"] = (
    train_clean["code"].progress_apply(remove_comments)
)

train_clean["code_length_no_comments"] = (
    train_clean["code_no_comments"].str.len()
)

Удаляем комментарии во всём train_clean:   0%|          | 0/11830994 [00:00<?, ?it/s]

### Пустые данные и дубли:

После первых махинацей над данными могли появиться первые плохие примеры, которые стоит дропнуть. Поищем их и отбросим при необходимости.

In [66]:
for column in ["question", "code_no_comments"]:
    empty = train_clean[column].fillna("").str.strip().eq("")
    print(f"{column}: {empty.sum():,} пустых из {len(train_clean):,}")

question: 0 пустых из 11,830,994
code_no_comments: 0 пустых из 11,830,994


In [67]:
duplicate_mask = train_clean.duplicated(
    subset=["question", "code_no_comments"],
    keep="first",
)

print(f"Полных дублей: {duplicate_mask.sum():,} ({duplicate_mask.mean():.2%})")

Полных дублей: 312,555 (2.64%)


Все таки дубли есть, дропним их.

In [68]:
n_before = len(train_clean)

train_clean = train_clean.drop_duplicates(
    subset=["question", "code_no_comments"],
    keep="first",
)

print(f"Удалено дублей: {n_before - len(train_clean):,}")
print(f"Осталось примеров: {len(train_clean):,}")

Удалено дублей: 312,555
Осталось примеров: 11,518,439


In [96]:
solutions_per_question = (
    train_clean
    .groupby("question", sort=False)
    .size()
    .rename("n_solutions")
)

print(f"Уникальных вопросов: {len(solutions_per_question):,}")

display(
    solutions_per_question
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .to_frame()
    .round(2)
)

distribution = (
    solutions_per_question
    .value_counts()
    .sort_index()
    .rename_axis("n_solutions")
    .to_frame("n_questions")
)

distribution["percent_questions"] = (
    distribution["n_questions"] / len(solutions_per_question) * 100
)

display(distribution.round(2))

Уникальных вопросов: 10,916,230


,n_solutions
count,10916230.00
mean,1.06
std,0.43
min,1.00
50%,1.00
90%,1.00
95%,1.00
99%,2.00
max,253.00


,n_questions,percent_questions
n_solutions,,
1,10437206,95.61
2,413521,3.79
3,50458,0.46
4,7459,0.07
5,2417,0.02
...,...,...
148,1,0.00
151,1,0.00
161,1,0.00


У нас еще и разных ответов для одного входа прилично. Надо это учесть при разбиени на train и val.

### Работоспособность кода

Последнее, чем хотелось бы заняться в разборе данных: фильтрацией неправильных примеров в traion. Понять, правильное ли решение у нас в входных данных стоит дорого (надо либо глазами смотреть, либо травить ллмку оценивать качество каждого примера, что оочень дорого и долго). Поэтому обойдемся простым запуском кода решения задач в изолированной среде, так мы точно выбросим неправильные примеры.

Для начала просто посмотрим на очень маленьком семпле, сколько таких программ вообще есть, чтобы оценить жизнеспособность этой идеи.

Соберем код для изолированных запусков питон скриптов.

In [91]:
import importlib
import pandas as pd
from time import perf_counter
from tqdm.auto import tqdm
from lib.utils import execution_check

importlib.reload(execution_check)

execution_sample = train_clean.sample(
    n=50_000,
    random_state=seed,
).copy()

started = perf_counter()

reports = pd.DataFrame(
    tqdm(
        execution_check.execute_solutions(
            execution_sample["code_no_comments"],
            timeout=2.0,
            workers=8,
            return_reports=True,
        ),
        total=len(execution_sample),
        desc="Проверяем solve()",
    ),
    index=execution_sample.index,
)

execution_sample = execution_sample.join(reports)

print(f"Время проверки: {perf_counter() - started:.1f} с")

Проверяем solve():   0%|          | 0/50000 [00:00<?, ?it/s]

Время проверки: 284.1 с


Посмотим на долю ошибочных программ:

In [92]:
summary = execution_sample["status"].value_counts().to_frame("execution_count")
summary["execution_percent"] = summary["execution_count"] / len(execution_sample) * 100

summary.round(2)

,execution_count,execution_percent
status,,
ok,49202,98.4
error,798,1.6


In [93]:
failed = execution_sample.loc[execution_sample["status"] != "ok"]

display(
    failed.groupby(["status", "error_type"])
    .size()
    .sort_values(ascending=False)
    .to_frame("count")
)

display(failed[["error_type", "error_message"]].head(10))

count
status error_type                
error  NameError              759
       ModuleNotFoundError     39

,error_type,error_message
11096324,ModuleNotFoundError,No module named 'sympy'
11529082,NameError,name 'math' is not defined
5324336,NameError,name 'math' is not defined
11770697,NameError,name 'math' is not defined
6100239,NameError,name 'math' is not defined
6833748,NameError,name 'math' is not defined
6879277,NameError,name 'math' is not defined
10634668,NameError,name 'math' is not defined
8341359,NameError,name 'math' is not defined
6280348,NameError,name 'math' is not defined


In [ ]:
print("Количество уникальных типов ошибок:", len(failed["error_type"].unique()))
print("Уникальные сообщения ошибок:")
for error_message in failed["error_message"].unique():
    print("-", error_message)

Количество уникальных типов ошибок: 2
Уникальные сообщения ошибок:
- No module named 'sympy'
- name 'math' is not defined
- name 'result' is not defined
- name 'sqrt' is not defined
- name 'pi' is not defined
- name 'ceil' is not defined
- name 'symbols' is not defined
- No module named 'scipy'
- name 'sympy' is not defined
- name 'e' is not defined


Посмотрим еще на выходные типы кода:

In [95]:
execution_sample.loc[execution_sample["status"] == "ok", "return_type"].value_counts().to_frame("count")

,count
return_type,
float,25831
int,22219
tuple,857
str,210
list,28
bool,25
NoneType,15
ellipsis,13
Fraction,2


Как видно, есть ошибочный код без импортов и такого примерно 1.5%. Но правильным образом вставлять `import math` (+ `from math import *` + `sympy`) и прочее в нужные решение нельзя, так как это генерация новых данных. А в рамках соревнования это запрещено, поэтому сможем только фильтровать и то не факт, что будем. По цпу компьюту дорого.

В ходе EDA изучили примеры задач и решений, распределения длин, выбросы, комментарии, дубли и количество решений на вопрос. Пустых вопросов и очищенных решений не обнаружено.

Для подготовки данных выбрали следующие правила:

- Оставлять вопросы длиной до **1000 символов** и исходный код до **1200 символов** включительно.
- Удалять комментарии через `tokenize`, сохраняя исходный код в отдельной колонке.
- Удалять полные дубли по паре `question` и `code_no_comments`. Исправление программ и добавление импортов не выполняем.

На случайной выборке из **50 000 программ** успешно выполнились **98,4%**. Ошибки были связаны с неопределёнными именами и отсутствующими пакетами. Полную фильтрацию train по исполнимости пока не будем делать из-за стоимости прогона.

Примерно у **4% вопросов** есть несколько решений. Поэтому train и validation будем разделять **по вопросам**: все решения одного вопроса должны попадать в один split. Для оценки можно использовать один выбранный эталон на вопрос, исключив из train все решения этого вопроса.